# M8b PCA 차원축소 — 실습 (W13, M8 2부작 완결편)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 4점 세트피스의 **분산 몰아주기**(원래 축 20/20 → 새 축 32/8)를 코드로 재현하고 **sklearn 검산**([0.8, 0.2])으로 일치를 확인한다 ⭐
2. digits **64차원 → 2D** 시각화(정보 21.6%)와 **누적 설명 분산**(31개 = 90%)을 실측한다
3. **재구성 실험**으로 압축을 눈으로 확인한다(2/10/30개 복원 이미지) ⭐

**7단계 멘탈모델 초점:** 표현(비지도) — 정답 없이 "줄이기"

## Part A. 세트피스 — 세상에서 가장 작은 PCA ⭐
점 4개 (3,1), (1,3), (−3,−1), (−1,−3). 원래 축의 정보 배분은 50:50인데, 대각선 축으로 갈아타면? 먼저 종이에서 제곱합을 계산해 보고, 코드로 검산하세요.

In [ ]:
import numpy as np                                     # 수치 계산

P = np.array([[3, 1], [1, 3], [-3, -1], [-1, -3]], dtype=float)  # 점 4개(평균 = 원점)
print('x 제곱합:', (P[:, 0] ** 2).sum(), '| y 제곱합:', (P[:, 1] ** 2).sum())  # 20 / 20 — 50:50

d1 = np.array([1, 1]) / np.sqrt(2)                     # 후보 새 축 PC1 = 대각선 (단위벡터)
d2 = np.array([1, -1]) / np.sqrt(2)                    # PC2 = 그와 수직
t1 = P @ ___                                           # ✍️ 빈칸: PC1으로의 투영 = 어느 방향과의 내적?
t2 = P @ d2                                            # PC2로의 투영
print('PC1 투영:', np.round(t1, 4), '→ 제곱합:', round((t1 ** 2).sum(), 4))   # ±2√2 → 32
print('PC2 투영:', np.round(t2, 4), '→ 제곱합:', round((t2 ** 2).sum(), 4))   # ±√2 → 8
print('배분:', round(32 / 40, 2), ':', round(8 / 40, 2))                      # 0.8 : 0.2 — 몰아주기!

In [ ]:
from sklearn.decomposition import PCA                  # 이제 sklearn으로 검산

pca = PCA(n_components=___).fit(P)                     # ✍️ 빈칸: 주성분 몇 개까지 보나? (2차원 데이터의 전부)
print('설명된 분산 비율:', np.round(pca.explained_variance_ratio_, 4))  # [0.8, 0.2] — 손계산 그대로?
print('PC1 방향:', np.round(pca.components_[0], 4))     # (0.707, 0.707) = (1,1)/√2 — 그 대각선?

> **검산 포인트:** 원래 축 **20/20(50:50)** → 새 축 **32/8(80:20)**, 합은 40으로 동일 — **총 정보는 그대로, 배분만 바꿨다.** sklearn도 [0.8, 0.2]·PC1 (0.707, 0.707)로 일치. PCA = 모든 방향 중 제곱합(분산) 최대를 PC1로, 수직 중 최대를 PC2로… 고르는 것. PC2를 버리면 2차원 → 1차원인데 정보 80% 생존.

## Part B. 실전 — 손글씨 64차원을 2D로
digits 1,797장(8×8=64차원)을 표준화 후 2개 주성분에 투영해 산점도로 봅니다. 정답 색칠은 **그림 확인용**일 뿐 — PCA는 y를 본 적이 없습니다(비지도!).

In [ ]:
import matplotlib.pyplot as plt                        # 그래프
from sklearn.datasets import load_digits               # 손글씨 숫자(8x8)
from sklearn.preprocessing import StandardScaler       # 표준화(분산은 단위 싸움 — M3·M8a와 같은 뿌리)

digits = load_digits()                                 # 1,797장
Xd = StandardScaler().fit_transform(digits.___)        # ✍️ 빈칸: 픽셀 행렬이 담긴 속성(64차원 X)
pca2 = PCA(n_components=2).fit(Xd)                     # 주성분 2개
proj = pca2.transform(Xd)                              # (1797, 64) → (1797, 2)
print('원래 차원:', Xd.shape[1], '→ 축소 차원:', proj.shape[1])
print('2개가 담은 정보:', round(pca2.explained_variance_ratio_.sum(), 3))  # 0.216 — 21.6%뿐!

plt.figure(figsize=(7, 6))
sc = plt.scatter(proj[:, 0], proj[:, 1], c=digits.target, cmap='tab10', s=15)  # 색 = 정답(확인용)
plt.colorbar(sc, label='digit')
plt.xlabel('PC1'); plt.ylabel('PC2')                   # 축(영어)
plt.title('Digits projected to 2D by PCA')
plt.show()

> **관찰:** 정답을 본 적 없는 PCA가 2차원에 눌러 그렸는데 같은 숫자끼리 어느 정도 모입니다 — 그것도 정보 **21.6%**(PC1 12.0% + PC2 9.6%)만으로. 겹침이 많은 것도 당연 — 나머지 78%가 없는 **요약본**이니까요. 그 손실을 다음 Part에서 숫자로 잽니다.

## Part C. 몇 개면 충분한가 — 누적 설명 분산
주성분 수를 늘리며 누적 비율을 보고, `PCA(n_components=0.90)`(비율 지정!)이 몇 개를 고르는지 확인합니다.

In [ ]:
pca10 = PCA(n_components=10).fit(Xd)                   # 주성분 10개
ratio = pca10.___                                      # ✍️ 빈칸: 각 주성분의 설명된 분산 비율 속성
print('각 비율:', np.round(ratio, 3))
print('상위 2 누적:', round(ratio[:2].sum(), 3), '| 상위 10 누적:', round(ratio.sum(), 3))  # 0.216 / 0.589

full = PCA().fit(Xd)                                   # 전부(64개) 학습
cum = np.cumsum(full.explained_variance_ratio_)        # 누적 곡선
plt.plot(range(1, 65), cum, '-')
plt.axhline(0.90, color='gray', ls='--', lw=1)         # 90% 선
plt.xlabel('number of components'); plt.ylabel('cumulative explained variance')
plt.grid(True, alpha=0.3)
plt.title('Cumulative explained variance (digits)')
plt.show()

p90 = PCA(n_components=___).fit(Xd)                    # ✍️ 빈칸: '개수'가 아니라 '보존할 비율'로 지정(90%)
print('90% 보존에 필요한 주성분 수:', p90.n_components_)  # 31 — 절반 이하로 90%!

> **관찰:** 누적 — 2개 21.6% · 8개 52.9%(여기서 절반 돌파) · 10개 58.9% · 20개 79.3% · 30개 89.3% · **31개 90%** · 64개 100%. 뒤쪽 33개의 몫은 10%뿐(잡음에 가까움). "몇 개"의 정답은 없고 **용도별 절충**(시각화 2, 압축·전처리는 곡선 보고 90~95% 선) — M8a 엘보우와 같은 종류의 판단.

## Part D. 재구성 — 압축을 눈으로 ⭐
줄였다가 `inverse_transform`으로 되돌려 원본과 비교합니다. (복원 **이미지**를 보기 위해 여기선 표준화 없이 **원본 픽셀**로 — 그래서 비율 수치가 Part C와 다릅니다.)

In [ ]:
Xraw = digits.data                                     # 원본 픽셀(0~16)
sample = Xraw[0:1]                                     # 샘플 한 장(숫자 0)
ncs = [2, 10, 30]                                      # 주성분 수 세 가지

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(sample.reshape(8, 8), cmap='gray_r')    # 원본
axes[0].set_title('original (64)')
for ax, nc in zip(axes[1:], ncs):                      # 각 압축 수준
    pr = PCA(n_components=nc).fit(Xraw)                # nc개 주성분 학습
    rec = pr.___(pr.transform(sample))                 # ✍️ 빈칸: 줄인 것을 되돌리는 메서드
    mse = np.mean((rec - sample) ** 2)                 # 복원 오차
    ax.imshow(rec.reshape(8, 8), cmap='gray_r')
    ax.set_title(f'{nc} comps (MSE {mse:.2f})')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

for nc in ncs + [64]:                                  # 오차·누적 비율 표
    pr = PCA(n_components=nc).fit(Xraw)
    rec = pr.inverse_transform(pr.transform(sample))
    print(f'n={nc:>2}: 복원 MSE {np.mean((rec - sample) ** 2):.2f} | 누적 비율 {pr.explained_variance_ratio_.sum():.3f}')

> **관찰:** 2개 = 얼룩(MSE 8.41) → 10개 = 알아볼 만(2.23) → **30개 = 원본과 구별 불가(0.37)** → 64개 = 완벽(0.00). **64개 숫자를 30개로 줄여 보관해도 눈은 못 느낀다** — 압축의 실용. (원본 픽셀 기준 누적: 2개 28.5% · 10개 73.8% · 30개 95.9% — Part C의 표준화 기준과 기준이 달라 숫자가 다름.) **2학기 예고:** "줄였다 복원하며 핵심만"의 신경망판·비선형판 = 오토인코더(2학기 D6).

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "내가 점 4개를 새로 만들어 제곱합 배분을 손으로 계산할 테니 채점해 줘."
- "'분산이 곧 정보'인 이유를 분산 0인 축이라는 반례로 설명해 볼게 — 허점을 찔러 줘."
- "2D 그림(21.6%)에서 숫자들이 겹치는 걸 어떻게 읽어야 하는지 말해 볼게."
- "PCA 전처리를 분할 전에 전체 데이터로 하면 왜 누수인지 M2a로 설명해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 4점 세트피스로 **분산 몰아주기**(20/20 → 32/8, 50:50 → 80:20)를 재현하고 sklearn 검산([0.8, 0.2]·PC1 (0.707, 0.707)) 일치를 확인했다
2. digits 64차원을 2D로(정보 21.6%로도 윤곽) 그리고, 누적 곡선에서 **31개 = 90%**(`PCA(0.90)`)를 읽었다
3. 재구성으로 압축을 눈으로 확인했다(30개면 원본과 구별 불가, MSE 0.37)

**스스로 점검**
- [ ] 4점의 제곱합 배분(20/20 → 32/8)을 종이에 재현할 수 있다
- [ ] "분산이 곧 정보"인 이유를 말할 수 있다
- [ ] 2D 산점도를 "요약본"으로 읽는 이유(21.6%)를 안다
- [ ] `PCA(n_components=0.90)`의 뜻과 digits에서의 답(31)을 안다
- [ ] PCA에 표준화가 필요한 이유를 M3·M8a와 연결할 수 있다

**🔹심화 (선택)**
- `pca2.components_[0]`을 8×8로 접어(`reshape(8, 8)`) 이미지로 그려 보세요 — "PC1이 보는 패턴"이 나타납니다.
- PCA(31)로 줄인 특징을 로지스틱 회귀(M5)에 넣어 64차원 원본과 정확도를 비교해 보세요 — 단, **분할 먼저·train으로만 PCA 학습**(M2a의 누수 원칙!).
- Part B를 표준화 없이 다시 돌려 보세요 — digits(픽셀 0~16 동일 범위)에서는 차이가 작지만, 왜 실전에선 표준화가 원칙인지 생각해 보세요.

**다음 시간(M9):** 1학기 피날레 — XOR과 "왜 신경망인가", 2학기로 가는 다리.